# SceneHawk: LLM labeling, spot-check, and split
Runs steps 3–5 of the labeling plan:
1. **Pilot** – the LLM labels the reviews you already hand-labeled, so you can see if it follows the guide before spending money.
2. **Big run** – the LLM labels ~4,000 new `unsup` reviews.
3. **Spot-check** – export 100 LLM-labeled reviews, label them blind by hand, measure agreement.
4. **Split** – learning / checking / exam sets, split by film so no movie leaks across sets.

Run cells top to bottom. Every LLM result is saved to a file as it arrives, so if Colab disconnects, re-run and it picks up where it stopped.

## 0. Setup
Add your OpenRouter key in Colab: 🔑 **Secrets** (left sidebar) → name it `OPENROUTER_API_KEY` → turn on notebook access.

In [ ]:
!pip -q install openai pandas scikit-learn
import os, json, re, random, glob, time, concurrent.futures as cf
import pandas as pd, numpy as np
from openai import OpenAI
from google.colab import userdata, files

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))

# Copy the exact model slug from openrouter.ai/models. Pick a cheap, strong model;
# check its price per million tokens there before the big run.
MODEL = "deepseek/deepseek-v4.1-flash"

FEELINGS = ["tense","dread","scared","unsettled","disturbed",
            "sad","grief-stricken","lonely","moved","bittersweet",
            "cozy","heartwarming","tender","hopeful","uplifted","joyful",
            "thrilled","amused","awed","curious",
            "thoughtful","nostalgic","inspired","confused",
            "uncomfortable","angry","disgusted","drained"]
assert len(FEELINGS) == 28
os.makedirs("work", exist_ok=True)

## 1. Load the hand labels
Upload all three teammates' CSVs (the full, corrected files, straight from your computer).

In [ ]:
uploaded = files.upload()   # pick roger's, mate's and apurba's CSVs

def load_labels(path):
    df = pd.read_csv(path)
    keep = [c for c in ["id","film","year","review","labeler"] + FEELINGS + ["notes"] if c in df.columns]
    df = df[keep].copy()
    df[FEELINGS] = df[FEELINGS].fillna(0).astype(int)
    df["labeler"] = df.get("labeler", pd.Series(dtype=str)).fillna(os.path.basename(path)).astype(str)
    return df.dropna(subset=["id", "review"])

gold = pd.concat([load_labels(p) for p in uploaded], ignore_index=True)
print(len(gold), "hand labels,", gold["id"].nunique(), "unique reviews")
print(gold.groupby("labeler")[FEELINGS].sum().sum(axis=1).rename("labels marked"))
print("share of blank rows per labeler:")
print(gold.assign(blank=gold[FEELINGS].sum(axis=1)==0).groupby("labeler")["blank"].mean().round(2))

Saving hand_labels_all.csv to hand_labels_all.csv
126 hand labels, 111 unique reviews
labeler
apurba    39
mate      35
roger     31
Name: labels marked, dtype: int64
share of blank rows per labeler:
labeler
apurba    0.47
mate      0.46
roger     0.57
Name: blank, dtype: float64


**Look at the blank-row share above.** The guide says many reviews should have no feelings. If one labeler has far fewer blank rows than the others, they are probably labeling quality judgments ("this was awful" → angry). Fix that before the pilot, because the LLM will be judged against these labels.

In [ ]:
# Free agreement check: reviews that more than one person labeled
def agreement(a, b):
    """a, b: 0/1 arrays (rows x feelings). Returns exact-row match and per-feeling F1."""
    exact = (a == b).all(axis=1).mean()
    tp = (a & b).sum(0); fp = (a & ~b).sum(0); fn = (~a & b).sum(0)
    f1 = 2*tp / np.maximum(2*tp + fp + fn, 1)
    return exact, pd.Series(f1, index=FEELINGS)

shared = gold.groupby("id").filter(lambda g: g["labeler"].nunique() > 1)
labelers = sorted(gold["labeler"].unique())
for i in range(len(labelers)):
    for j in range(i+1, len(labelers)):
        A = shared[shared.labeler==labelers[i]].set_index("id")[FEELINGS]
        B = shared[shared.labeler==labelers[j]].set_index("id")[FEELINGS]
        common = A.index.intersection(B.index)
        if len(common) == 0: continue
        A, B = A.loc[common].values.astype(bool), B.loc[common].values.astype(bool)
        exact, f1 = agreement(A, B)
        print(f"{labelers[i]} vs {labelers[j]}: {len(common)} shared reviews, exact-row match {exact:.0%}")

apurba vs mate: 8 shared reviews, exact-row match 100%
apurba vs roger: 5 shared reviews, exact-row match 100%
mate vs roger: 6 shared reviews, exact-row match 100%


## 2. Load the review corpus (aclImdb `unsup`)
Downloads the dataset (~80 MB) and builds ids like `train-unsup-21400`, matching your CSVs. `imdb_id` is the film, used later so the same film never lands in two sets.

In [ ]:
gold.loc[gold["id"] == "train-unsup-8093", "curious"] = 1

In [ ]:
if not os.path.exists("aclImdb"):
    !wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz -O aclImdb_v1.tar.gz
    if os.path.getsize("aclImdb_v1.tar.gz") < 80_000_000:
        print("Download failed. Pick your aclImdb_v1_tar.gz file instead:")
        up = files.upload()
        os.replace(list(up)[0], "aclImdb_v1.tar.gz")
    !tar -xzf aclImdb_v1.tar.gz

urls = open("aclImdb/train/urls_unsup.txt").read().split()
rows = []
for path in glob.glob("aclImdb/train/unsup/*.txt"):
    n = int(os.path.basename(path).split("_")[0])
    text = open(path, encoding="utf-8").read().replace("<br />", " ")
    rows.append({"id": f"train-unsup-{n}", "imdb_id": re.search(r"tt\d+", urls[n]).group(0), "review": text})
corpus = pd.DataFrame(rows)
print(len(corpus), "reviews,", corpus.imdb_id.nunique(), "films")
gold = gold.merge(corpus[["id","imdb_id"]], on="id", how="left")

--2026-10-24 00:39:52--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... failed: Connection refused.
Download failed. Pick your aclImdb_v1_tar.gz file instead:


Saving aclImdb_v1.tar.gz to aclImdb_v1.tar (1).gz
50000 reviews, 7091 films


## 3. The labeling prompt
This is the guide turned into instructions. **Paste the definitions from `feelings-list.md`** into `DEFINITIONS` so the LLM uses the same meanings your team agreed on.
The model must quote the words that justify each feeling. That stops it from guessing, the same way the guide says "label only what the text supports".

In [ ]:
DEFINITIONS = """
tense: a sustained edge; unable to relax while watching
dread: slow certainty that something bad is coming
scared: sudden fright, jump scares
unsettled: uneasy, something feels off or creepy
disturbed: shaken by something dark or cruel that stays with you
sad: feeling down or close to tears
grief-stricken: deep sorrow over loss or death
lonely: a sense of isolation or emptiness
moved: emotionally touched; it got to you
bittersweet: happy and sad at once
cozy: warm, safe, comfortable, like a blanket
heartwarming: warmth from kindness or love on screen
tender: gentle affection, intimacy
hopeful: believing things can get better
uplifted: lifted up, left feeling better than before
joyful: delight, happiness, pure fun
thrilled: excited, adrenaline, edge-of-seat in a good way
amused: laughing, finding it funny (including laughing at a bad film)
awed: overwhelmed by spectacle, scale or wonder
curious: wanting to know what happens or what it means
thoughtful: made to think or reflect
nostalgic: taken back to one's own past
inspired: motivated to act or change something in one's life
confused: intended disorientation, not knowing what is real (not bad writing)
uncomfortable: squirming, awkward, wanting to look away
angry: furious at injustice or cruelty shown (not at the film's quality)
disgusted: revolted, grossed out
drained: emotionally exhausted by the film (not bored)
"""

SYSTEM = f"""You label IMDb film reviews for SceneHawk.

Task: mark which feelings the FILM gave the VIEWER while they watched it.

Rules:
1. Label what the viewer felt while watching, not whether they liked the film.
2. These are NOT feelings and get no label: quality judgments ("awful acting", "10/10"),
   plot summary, craft admiration ("beautiful cinematography").
3. Many reviews have NO feelings. An empty list is normal and correct. Never stretch:
   "the film was bad" is not sad; "great acting" is not moved.
4. Anger: "it made me furious how they were treated" = angry. Angry that the film was bad = no label.
5. scared = sudden fright. tense = sustained edge. dread = slow certainty something bad is coming.
6. confused = an intended effect ("I never knew what was real"). "The plot made no sense" = no label.
7. Sarcasm: "I cried laughing" = amused, not sad.
8. Second-hand reports count: "my daughter's heart was pounding" is evidence about the film.
9. Use only the text. Ignore what you know about the film.
10. Boredom is not a feeling: "boring", "almost fell asleep" = no label. drained = emotionally worn out by the film.
11. Laughing at a bad film counts: "so bad I had to laugh" = amused. Just calling it "laughable" = no label.
12. Generic enjoyment ("enjoyed it", "fun", "entertaining") = no label unless a specific feeling is shown.
13. Effect words COUNT and must be labeled. When the review calls the film funny/humorous/hilarious -> amused; exciting/thrilling/gripping -> thrilled; suspenseful -> tense; feel-good/happy ending -> heartwarming or joyful; powerful/emotional/touching -> moved; poignant -> bittersweet; jumpy -> scared; horrifying/gross -> disgusted. Only words about how well it was made are quality words (great, awful, well-acted, boring).
13b. Label EVERY feeling that has evidence, not just the strongest one. Anger or disgust at bad changes, bad acting or wasted money ("biting my lip", "made me feel sick at what they added") is a quality complaint: no label.
14. awed = overwhelmed by the experience or spectacle. Admiring a performance or technique = no label.
15. Anger at what the film says or shows (propaganda, cruelty, injustice) = angry. Angry the film is bad = no label.
16. nostalgic = the film takes the viewer back to their own past. A film merely being old = no label.
17. A character's feelings in a plot summary are not the viewer's feelings.
18. Every feeling needs a quote from the review. No quote, no label.
19. If you are torn on a feeling, leave it out and set "unsure": true with a short note.

Allowed feelings (use these exact words only): {", ".join(FEELINGS)}

Definitions:
{DEFINITIONS}

Reply with JSON only:
{{"feelings": [...], "evidence": {{"<feeling>": "<short quote from the review>"}}, "unsure": false, "note": ""}}"""

# Worked examples from the team guide (not from the hand-labeled files, so the pilot stays fair)
FEWSHOT = [
 ("A really fun, funny little film with a happy ending. It's the kind of feel-good movie the whole family can enjoy.",
  {"feelings": ["amused", "joyful", "heartwarming"], "evidence": {"amused": "funny", "joyful": "a happy ending", "heartwarming": "feel-good movie the whole family can enjoy"}, "unsure": False, "note": "effect words count (rule 13)"}),
 ("They ruined the book. Every change they made had me grinding my teeth and wanting to throw something at the screen. Avoid.",
  {"feelings": [], "evidence": {}, "unsure": False, "note": "anger at a bad adaptation is a quality complaint"}),
 ("I heard this film had got Oscar for something... Extremely unimpressive McQuinn's play and Hoffmann who tries to save the situation and fails.",
  {"feelings": [], "evidence": {}, "unsure": False, "note": "quality verdict only"}),
 ("Clint Eastwood stars as Wes Block a police officer who hunts a serial killer... Some good acting from Eastwood and Bujold elevate this one above mediocrity.",
  {"feelings": [], "evidence": {}, "unsure": False, "note": "plot summary and quality judgment"}),
 ("...he runs around being BADASS... This film is basically ninjas (rubbish ninjas) getting beaten up by little kids for ages... watch this movie because it's got Hulk Hogan in it.",
  {"feelings": ["amused"], "evidence": {"amused": "ninjas (rubbish ninjas) getting beaten up by little kids"}, "unsure": False, "note": ""}),
 ("We watched it with my son. He was gripping the armrest the whole last act, and by the end we were both wiped out but grinning.",
  {"feelings": ["tense", "drained", "joyful"], "evidence": {"tense": "gripping the armrest the whole last act", "drained": "wiped out", "joyful": "grinning"}, "unsure": False, "note": "second-hand report counts"}),
]

def build_messages(review):
    msgs = [{"role": "system", "content": SYSTEM}]
    for text, ans in FEWSHOT:
        msgs += [{"role": "user", "content": f"Review:\n{text}"},
                 {"role": "assistant", "content": json.dumps(ans)}]
    msgs.append({"role": "user", "content": f"Review:\n{review[:6000]}"})
    return msgs

def parse(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    d = json.loads(m.group(0))
    d["feelings"] = [f for f in d.get("feelings", []) if f in FEELINGS]   # drop invented words
    return d

def label_one(rid, review, tries=4):
    err = None
    for t in range(tries):
        try:
            r = client.chat.completions.create(model=MODEL, messages=build_messages(review),
                                               temperature=0, max_tokens=400)
            d = parse(r.choices[0].message.content)
            d["id"] = rid
            return d
        except Exception as e:
            err = e
            time.sleep(2 ** t)
    return {"id": rid, "feelings": None, "error": str(err)}

def label_many(df, out_path, workers=8):
    """Labels every row of df (needs id, review). Appends to out_path; skips ids already done."""
    done = set()
    if os.path.exists(out_path):
        done = {json.loads(l)["id"] for l in open(out_path) if '"feelings": null' not in l}
    todo = df[~df["id"].isin(done)]
    print(f"{len(done)} already done, {len(todo)} to go")
    with open(out_path, "a") as f, cf.ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(label_one, r.id, r.review) for r in todo.itertuples()]
        for k, fut in enumerate(cf.as_completed(futs), 1):
            f.write(json.dumps(fut.result()) + "\n"); f.flush()
            if k % 100 == 0: print(k, "labeled")

def read_results(path):
    res = [json.loads(l) for l in open(path)]
    res = [r for r in res if r.get("feelings") is not None]
    out = pd.DataFrame({"id": [r["id"] for r in res]})
    for f in FEELINGS:
        out[f] = [int(f in r["feelings"]) for r in res]
    out["llm_unsure"] = [r.get("unsure", False) for r in res]
    out["llm_evidence"] = [json.dumps(r.get("evidence", {})) for r in res]
    return out.drop_duplicates("id", keep="last")

print(label_one("test", "I laughed so hard I cried, and the ending had me hugging my sister."))

{'feelings': ['amused', 'heartwarming'], 'evidence': {'amused': 'I laughed so hard I cried', 'heartwarming': 'the ending had me hugging my sister'}, 'unsure': False, 'note': '', 'id': 'test'}


## 4. Pilot: does the LLM agree with your hand labels?
Costs a few cents. Do **not** skip this. It is the only proof the LLM labels can be trusted.

In [ ]:
pilot_in = gold.drop_duplicates("id")[["id","review"]]
label_many(pilot_in, "work/pilot.jsonl")
llm = read_results("work/pilot.jsonl").set_index("id")

for who in labelers:
    H = gold[gold.labeler==who].drop_duplicates("id").set_index("id")[FEELINGS]
    common = H.index.intersection(llm.index)
    exact, f1 = agreement(llm.loc[common, FEELINGS].values.astype(bool), H.loc[common].values.astype(bool))
    print(f"LLM vs {who}: {len(common)} reviews, exact-row match {exact:.0%}, "
          f"LLM blank {(llm.loc[common, FEELINGS].sum(1)==0).mean():.0%} vs human blank {(H.loc[common].sum(1)==0).mean():.0%}")

0 already done, 111 to go
100 labeled
LLM vs apurba: 41 reviews, exact-row match 76%, LLM blank 56% vs human blank 49%
LLM vs mate: 38 reviews, exact-row match 74%, LLM blank 58% vs human blank 47%
LLM vs roger: 42 reviews, exact-row match 93%, LLM blank 57% vs human blank 60%


In [ ]:
# Read the disagreements. For each, decide: was the human wrong, the LLM wrong, or the guide unclear?
g = gold.drop_duplicates(["id","labeler"]).set_index("id")
show = []
for rid in llm.index.intersection(g.index):
    human = sorted({f for _, r in g.loc[[rid]].iterrows() for f in FEELINGS if r[f]==1})
    model = [f for f in FEELINGS if llm.loc[rid, f]==1]
    if set(human) != set(model):
        show.append({"id": rid, "human (any labeler)": human, "llm": model,
                     "evidence": llm.loc[rid, "llm_evidence"], "review": g.loc[[rid]].iloc[0]["review"][:300]})
pd.set_option("display.max_colwidth", 300)
pd.DataFrame(show).head(30)

,id,human (any labeler),llm,evidence,review
0,train-unsup-11976,"[amused, joyful]",[amused],"{""amused"": ""the knife scene almost killed me""}","This is one of the most richly imaginative, creative, downright enjoyable movies I've ever seen. I haven't enjoyed myself this much since Pulp Fiction. It took me a while to figure out who the hero was. First I thought it was the barber guy who got himself constantly smacked about by the landlad..."
1,train-unsup-20209,"[awed, disgusted, thoughtful]","[disturbed, disgusted]","{""disgusted"": ""stomach churning, twisted scenes"", ""disturbed"": ""some of the ugliest and darkest places of the human mind""}","Gyorgy Palfi's second feature Taxidermia is definitely a milestone in Hungarian film-making, it is a truly astonishing experience and I would thoroughly recommend it to anyone wanting to broaden their taste for cinema. I found the film to be a deep black comedy with some stomach churning, twiste..."
2,train-unsup-11955,"[amused, joyful]",[amused],"{""amused"": ""There was laughter abound. It is one of the funniest movies I've seen in a long time.""}","OK. So you've seen a lot of movies. If that's true you know most of the genres just keep imitating each other and there is a serious lack of originality. However, when watching Kung Fu Hussle, the only thing I can use to describe it is ""original."" It takes from a lot of different ideas but the o..."
3,train-unsup-12373,[awed],[],{},"I love this movie. Every time I hit it channel-surfing I think I'll watch it for just a minute and I end up watching the whole damn thing. Some of the best movies take one or two freak things and then play them out absolutely straight (e.g., Big, Back to the Future, Groundhog Day). Modern aircra..."
4,train-unsup-45880,[],[amused],"{""amused"": ""a couple of Williams' one liners are moderately funny""}","Robin Williams is one of the funniest people of all time. All time! When he's being interviewed, I cry with laughter. And he's made some excellent comedies over the years, such as ""Good Morning, Vietnam."" What's he doing in this junk? For that matter, what about the other actors? John Krasinki i..."
5,train-unsup-30907,"[tense, unsettled]",[tense],"{""tense"": ""Ghosthouse has more suspense than massacre""}","Ghosthouse - La Casa 3 (aka House: La Maldición Final) was released in the eighties when the House franchise was a pretty big business, not so big as the Friday the 13th or Nightmare on Elm Street franchises but something important. Fortunately Umberto Lenzi only tries to exploit this franchise ..."
6,train-unsup-18857,"[disgusted, disturbed, uncomfortable, unsettled]","[unsettled, disturbed, disgusted]","{""unsettled"": ""unnerving"", ""disturbed"": ""disturbing... left me feeling dirty"", ""disgusted"": ""disgusting experience""}","I love Rob Zombie's music and was looking forward to this movie when I first heard about it. From the recurring themes in his music, I thought it would be a bit like Evil Dead II: surreal, funny, supernatural, and purposefully cheesy. Instead, I got 90 minutes of torture porn. Zombie makes no at..."
7,train-unsup-31431,"[amused, disgusted]",[amused],"{""amused"": ""I did laugh along with Ash and the deer head""}","In the middle of the movie,some totally new characters show up and it's pretty obvious they're only there to get killed by the demons.But then again,what other purposes can characters in a horror movie have anyway?This second movie in the ""Evil Dead""-series is kinda different actually,it gets mo..."
8,train-unsup-43494,[amused],[],{},"Scared to death is a fun movie. I really liked the character of Bill Raymond (played by Nat Pendleton), who was a detective sitting around a house waiting for someone to be murdered so that he could impress his superiors on the police force. There were a great deal of fun turns in the movie. The..."
9,train-unsup-9912,"[tense, thrilled]",[tense],"{""tense"": ""tension you could cut with a knife ... excruciatingly long, nerve-

**When to move on:** the LLM should agree with each human about as well as the humans agree with each other (the numbers from step 1). If it keeps making the same mistake, add one rule or one few-shot example to `SYSTEM`/`FEWSHOT`, delete `work/pilot.jsonl`, and re-run the pilot. Two or three rounds is normal.

## 5. Big run: label ~4,000 new reviews

In [ ]:
N = 4000
gold_ids = set(gold["id"])
pool = corpus[~corpus["id"].isin(gold_ids)]
big_in = pool.sample(N, random_state=42)

# Rough size check before you spend money (about 4 characters per token)
prompt_tokens = len(json.dumps(build_messages(""))) / 4
review_tokens = big_in["review"].str.len().clip(upper=6000).mean() / 4
total = N * (prompt_tokens + review_tokens)
print(f"≈ {total/1e6:.1f} M input tokens + ≈ {N*120/1e6:.1f} M output tokens. Multiply by the model's price on openrouter.ai.")

≈ 8.2 M input tokens + ≈ 0.5 M output tokens. Multiply by the model's price on openrouter.ai.


In [ ]:
RUN_IT = True   # set True once the pilot looks good and the cost is OK
if RUN_IT:
    label_many(big_in, "work/big.jsonl", workers=8)
big = read_results("work/big.jsonl").merge(corpus, on="id") if os.path.exists("work/big.jsonl") else None
if big is not None:
    print(len(big), "labeled;", f"{(big[FEELINGS].sum(1)==0).mean():.0%} blank;", f"{big.llm_unsure.mean():.0%} marked unsure")
    print(big[FEELINGS].sum().sort_values(ascending=False))

3840 already done, 160 to go
100 labeled
3898 labeled; 61% blank; 11% marked unsure
amused            746
moved             209
nostalgic         133
thoughtful        132
tense             112
thrilled          105
angry              93
sad                90
unsettled          77
curious            70
scared             65
disturbed          50
disgusted          46
heartwarming       42
inspired           42
joyful             42
awed               39
uncomfortable      28
bittersweet        23
uplifted           19
hopeful            14
tender             10
confused           10
dread               9
cozy                5
grief-stricken      4
drained             4
lonely              0
dtype: int64


Check the counts per feeling. Any feeling with almost no examples (say under 30) will be hard for the model to learn. Note those for your report rather than forcing more labels.

## 6. Spot-check 100 by hand (blind)
The export has **no LLM labels in it**, so whoever checks isn't nudged by them. Open it in Google Sheets, put `1` under each feeling, upload it back.

In [ ]:
spot = big.sample(100, random_state=7)
blank_sheet = spot[["id","review"]].copy()
for f in FEELINGS: blank_sheet[f] = ""
blank_sheet["notes"] = ""
blank_sheet.to_csv("spotcheck_100_blank.csv", index=False)
files.download("spotcheck_100_blank.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
up = files.upload()            # the filled-in spotcheck CSV
human_spot = load_labels(list(up)[0]).set_index("id")
llm_spot = big.set_index("id").loc[human_spot.index, FEELINGS]
exact, f1 = agreement(llm_spot.values.astype(bool), human_spot[FEELINGS].values.astype(bool))
print(f"Spot-check: exact-row match {exact:.0%}")
print("Per-feeling F1 (feelings that appeared):")
print(f1[(human_spot[FEELINGS].sum() + llm_spot.sum()) > 0].round(2).sort_values())

Saving spotcheck_100_second_model.csv to spotcheck_100_second_model.csv
Spot-check: exact-row match 78%
Per-feeling F1 (feelings that appeared):
curious          0.00
disgusted        0.00
awed             0.50
moved            0.50
angry            0.57
sad              0.67
heartwarming     0.67
tense            0.67
uncomfortable    0.67
nostalgic        0.67
disturbed        0.67
unsettled        0.67
thoughtful       0.75
thrilled         0.80
amused           0.86
scared           1.00
inspired         1.00
joyful           1.00
dtype: float64


Write these numbers in your report: they are the honest quality of your LLM labels. If one feeling scores very low, either fix the prompt and relabel, or drop that feeling from training and say so.

## 7. Split into learning / checking / exam
- **exam** = only human labels: your hand-labeled reviews and the 100 spot-checks. Never train on these.
- **learning** and **checking** = LLM labels, 85% / 15%.
- Split **by film**: if a film is in the exam set, none of its reviews go into learning or checking. Otherwise the model can memorize a movie instead of learning feelings.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Exam: when several people labeled a review, a feeling counts if at least half of them marked it
exam_gold = gold.groupby(["id","imdb_id","review"], as_index=False)[FEELINGS].mean()
exam_gold[FEELINGS] = (exam_gold[FEELINGS] >= 0.5).astype(int)
exam_spot = human_spot.reset_index()[["id"] + FEELINGS].merge(corpus, on="id")
exam = pd.concat([exam_gold, exam_spot[exam_gold.columns]], ignore_index=True).drop_duplicates("id")

llm_rest = big[~big["id"].isin(exam["id"]) & ~big["imdb_id"].isin(exam["imdb_id"])]
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr, va = next(gss.split(llm_rest, groups=llm_rest["imdb_id"]))
learning, checking = llm_rest.iloc[tr], llm_rest.iloc[va]

cols = ["id","imdb_id","review"] + FEELINGS
for name, df in [("learning", learning), ("checking", checking), ("exam", exam)]:
    df[cols].to_csv(f"{name}.csv", index=False)
    print(f"{name}: {len(df)} reviews, {df.imdb_id.nunique()} films")
assert not set(learning.imdb_id) & set(exam.imdb_id) and not set(checking.imdb_id) & set(exam.imdb_id)
assert not set(learning.imdb_id) & set(checking.imdb_id)
print("No film appears in two sets.")
for n in ["learning","checking","exam"]: files.download(f"{n}.csv")

learning: 3029 reviews, 1994 films
checking: 534 reviews, 352 films
exam: 211 reviews, 167 films
No film appears in two sets.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>